Create paths and retrieve data

In [1]:
import os

# ÄNDRA denna så att den matchar din lokala struktur
BASE_DIR = "Project_data_2026"   # mappen som innehåller excel-filerna

PATH_FACTORS = os.path.join(BASE_DIR, "project Factors_2026.xlsx")
PATH_STOCKS  = os.path.join(BASE_DIR, "project Stocks_2026.xlsx")
PATH_FUNDS   = os.path.join(BASE_DIR, "project Funds_2026.xlsx")

print(PATH_FACTORS)
print(PATH_STOCKS)
print(PATH_FUNDS)

Project_data_2026/project Factors_2026.xlsx
Project_data_2026/project Stocks_2026.xlsx
Project_data_2026/project Funds_2026.xlsx


In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PATH_FACTORS = "Project_data_2026/Factors_2026.xlsx"

# --- Inspektera filen (sheets) ---
xls = pd.ExcelFile(PATH_FACTORS)
print("Sheets i Factors-filen:", xls.sheet_names)

# --- Load (försök 'factors', annars första sheet) ---
sheet = "factors" if "factors" in xls.sheet_names else xls.sheet_names[0]
f = pd.read_excel(PATH_FACTORS, sheet_name=sheet).copy()
print("Använder sheet:", sheet)
print("\nKolumner i data:")
print(list(f.columns))

# --- Standardisera kolumnnamn (strip/low) för att hitta rätt ---
cols = {c: c.strip() for c in f.columns}
f.rename(columns=cols, inplace=True)

# Hjälpfunktion för att hitta kolumn med flera möjliga namn
def pick_col(df, candidates):
    cand_lower = {c.lower(): c for c in df.columns}
    for name in candidates:
        if name.lower() in cand_lower:
            return cand_lower[name.lower()]
    return None

col_year  = pick_col(f, ["year", "Year"])
col_month = pick_col(f, ["month", "Month"])
col_date  = pick_col(f, ["Date", "date"])

col_rf    = pick_col(f, ["rf", "RF", "r_f"])
col_mktrf = pick_col(f, ["rm_rf", "Rm-Rf", "mktrf", "market_excess", "rmrf"])
col_smb   = pick_col(f, ["smb_vw", "SMB", "smb"])
col_hml   = pick_col(f, ["hml_vw", "HML", "hml"])

needed = {"year": col_year, "month": col_month, "Date": col_date,
          "rf": col_rf, "rm_rf": col_mktrf, "smb_vw": col_smb, "hml_vw": col_hml}

print("\nMatchade kolumner:")
for k, v in needed.items():
    print(f"{k:8s} -> {v}")

missing = [k for k,v in needed.items() if v is None]
if missing:
    raise ValueError(f"Saknar dessa nödvändiga kolumner i Factors-datan: {missing}. "
                     f"Kolla kolumnlistan ovan och uppdatera candidates.")

# --- Bygg standardiserad DF med exakt de namn vi använder vidare ---
ff = pd.DataFrame({
    "year":  f[col_year].astype(int),
    "month": f[col_month].astype(int),
    "Date":  pd.to_datetime(f[col_date], errors="coerce"),
    "rf":    pd.to_numeric(f[col_rf], errors="coerce"),
    "rm_rf": pd.to_numeric(f[col_mktrf], errors="coerce"),
    "smb_vw":pd.to_numeric(f[col_smb], errors="coerce"),
    "hml_vw":pd.to_numeric(f[col_hml], errors="coerce"),
}).dropna(subset=["Date"])

# --- Annualisera (multiplicera med 12 enligt instruktion) ---
for col in ["rf", "rm_rf", "smb_vw", "hml_vw"]:
    ff[col + "_ann"] = 12.0 * ff[col]

# --- Plotta faktorer ---
plt.figure()
plt.plot(ff["Date"], ff["rm_rf_ann"], label="Market excess (Rm-Rf)")
plt.plot(ff["Date"], ff["smb_vw_ann"], label="SMB")
plt.plot(ff["Date"], ff["hml_vw_ann"], label="HML")
plt.title("Faktorer (annualiserade)")
plt.xlabel("Date")
plt.ylabel("Return")
plt.legend()
plt.tight_layout()
plt.show()

# --- Medel & varians-kovarians ---
factor_cols = ["rm_rf_ann", "smb_vw_ann", "hml_vw_ann"]
print("Medel (annualiserade faktorer):")
print(ff[factor_cols].mean())

print("\nVarians-kovarians (annualiserade faktorer):")
print(ff[factor_cols].cov())

# --- Riskfri ränta ---
plt.figure()
plt.plot(ff["Date"], ff["rf_ann"], label="Rf")
plt.title("Riskfri ränta (annualiserad)")
plt.xlabel("Date")
plt.ylabel("Rf")
plt.legend()
plt.tight_layout()
plt.show()

print("\nGenomsnittlig riskfri ränta (annualiserad):", ff["rf_ann"].mean())


FileNotFoundError: [Errno 2] No such file or directory: 'Project_data_2026/Factors_2026.xlsx'